# 🛍️ 07: E-Commerce Omnichannel Retail Lakehouse & SCD Type 2 Pipeline

This notebook demonstrates the **Multi-Domain E-Commerce Medallion Architecture** featuring:
1. **Bronze Ingestion**: Ingesting 5 relational transactional tables (`orders`, `order_details`, `products`, `customers`, `categories`).
2. **Silver Normalization**: Date parsing, fulfillment duration math, and net line-item revenue calculations.
3. **Gold SCD Type 2 Modeling**: Slowly Changing Dimension (`dim_customer_scd2`) tracking customer loyalty tier upgrades and address history over time.
4. **Gold Star Schema Fact Tables**: `fct_orders` and `fct_order_items` joined to conformed dimensions.
5. **Customer 360 & RFM Segmentation**: Recency, Frequency, Monetary scoring and behavioral segmentation.
6. **DuckDB SQL Analytics**: Market basket and product category margin analysis.

In [ ]:
import os
import pandas as pd
import duckdb
from medallion.ecommerce_medallion import EcommerceMedallionPipeline

pipeline = EcommerceMedallionPipeline()
print("[+] Initialized E-Commerce Pipeline!")
print("    - Raw Directory:       ", pipeline.raw_dir)
print("    - Processed Directory: ", pipeline.processed_dir)
print("    - Warehouse Directory: ", pipeline.warehouse_dir)

## 🥉 1. Bronze Layer: Ingestion & Schema Profiling

In [ ]:
bronze_data = pipeline.run_bronze()
for tbl_name, df in bronze_data.items():
    print(f"Bronze Table `{tbl_name:<15}`: {len(df):>5} rows | Columns: {list(df.columns[:5])}...")

## 🥈 2. Silver Layer: Cleansing, Margin Math & Date Calculations

In [ ]:
silver_data = pipeline.run_silver(bronze_data)
df_silver_orders = silver_data["silver_orders"]
print("Silver Orders Sample (Showing Fulfillment & Net Totals):")
display(df_silver_orders[["orderID", "customerID", "orderDate", "shippedDate", "fulfillment_days", "total_gross_amount", "total_net_amount"]].head())

## 🥇 3. Gold Layer: SCD Type 2 Customer Dimension (`dim_customer_scd2`)

Notice how customer records track loyalty tier progression ('Bronze' -> 'Silver' -> 'Gold' -> 'Platinum')
with `effective_start_date`, `effective_end_date`, and `is_current_flag`.

In [ ]:
gold_data = pipeline.run_gold(silver_data)
df_scd2 = gold_data["dim_customer_scd2"]

# Filter a high-spending customer to see their historical versions
sample_cid = df_scd2[df_scd2["loyalty_tier"] == "Platinum"]["customer_id"].iloc[0]
print(f"=== SCD Type 2 History for Customer `{sample_cid}` ===")
display(df_scd2[df_scd2["customer_id"] == sample_cid][["customer_sk", "customer_id", "company_name", "loyalty_tier", "effective_start_date", "effective_end_date", "is_current_flag"]])

## 📊 4. Customer 360 & RFM Segmentation (`agg_customer_rfm`)

In [ ]:
df_rfm = gold_data["agg_customer_rfm"]
print("=== Customer Segment Distribution ===")
print(df_rfm["customer_segment"].value_counts().to_string())

print("\n=== Top Champions Segment Sample ===")
display(df_rfm[df_rfm["customer_segment"] == "Champions"][["customer_id", "company_name", "country", "recency_days", "frequency_orders_count", "monetary_total_spend", "customer_segment"]].head())

## 🦆 5. High-Performance DuckDB SQL Analytics over E-Commerce Tables

In [ ]:
con = duckdb.connect()
wh = pipeline.warehouse_dir.replace('\\', '/')

con.execute(f"CREATE TABLE dim_cust AS SELECT * FROM read_csv_auto('{wh}/dim_customer_scd2.csv')")
con.execute(f"CREATE TABLE dim_prod AS SELECT * FROM read_csv_auto('{wh}/dim_product.csv')")
con.execute(f"CREATE TABLE fct_ord  AS SELECT * FROM read_csv_auto('{wh}/fct_orders.csv')")
con.execute(f"CREATE TABLE fct_item AS SELECT * FROM read_csv_auto('{wh}/fct_order_items.csv')")

print("=== Top 5 Product Categories by Net Revenue & Margin ===")
q1 = """
    SELECT 
        p.category_name,
        COUNT(DISTINCT i.order_id) AS total_orders,
        SUM(i.quantity) AS total_units_sold,
        ROUND(SUM(i.line_gross_amount), 2) AS gross_revenue,
        ROUND(SUM(i.discount_amount), 2) AS total_discounts_given,
        ROUND(SUM(i.line_net_amount), 2) AS net_revenue
    FROM fct_item i
    JOIN dim_prod p ON i.product_sk = p.product_sk
    GROUP BY p.category_name
    ORDER BY net_revenue DESC
    LIMIT 5;
"""
print(con.execute(q1).df().to_string(index=False))

print("\n=== Average Fulfillment Latency by Shipping Country ===")
q2 = """
    SELECT 
        ship_country,
        COUNT(*) AS orders_shipped,
        ROUND(AVG(fulfillment_days), 1) AS avg_fulfillment_days,
        ROUND(SUM(freight), 2) AS total_freight_cost
    FROM fct_ord
    WHERE is_shipped = true AND fulfillment_days >= 0
    GROUP BY ship_country
    ORDER BY orders_shipped DESC
    LIMIT 8;
"""
print(con.execute(q2).df().to_string(index=False))